In [1]:
import pandas as pd
pd.set_option("display.max_columns", None)

In [ ]:
columns = [
    "age", "workclass", "fnlwgt", "education", "education-num",
    "marital-status", "occupation", "relationship", "race", "sex",
    "capital-gain", "capital-loss", "hours-per-week", "native-country", "income"
]

adult = pd.read_csv(
    r"E:\DSDA\CensusDataset\adult.data",
    header=None,
    names=columns,
    skipinitialspace=True
)

In [ ]:
adult_test = pd.read_csv(
    r"E:\DSDA\CensusDataset\adult.test",
    header=None,
    names=columns,
    skipinitialspace=True,
    comment="|"
)

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,25,Private,226802,11th,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,<=50K.
1,38,Private,89814,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States,<=50K.
2,28,Local-gov,336951,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States,>50K.
3,44,Private,160323,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States,>50K.
4,18,?,103497,Some-college,10,Never-married,?,Own-child,White,Female,0,0,30,United-States,<=50K.


In [7]:
print("Training file:", adult.shape[0], "rows and", adult.shape[1], "columns")
print("Test file:", adult_test.shape[0], "rows and", adult_test.shape[1], "columns")

Training file: 32561 rows and 15 columns
Test file: 16281 rows and 15 columns


In [ ]:
# Check for Null Values
print("Standard NaN counts:")
print(adult.isnull().sum())

print("\n'?' counts per column:")
print((adult == "?").sum())

missing_combined = adult.isnull().sum() + (adult == "?").sum()
print("\nTotal missing (NaN + '?') per column:")
print(missing_combined[missing_combined > 0])

Standard NaN counts:
age               0
workclass         0
fnlwgt            0
education         0
education-num     0
marital-status    0
occupation        0
relationship      0
race              0
sex               0
capital-gain      0
capital-loss      0
hours-per-week    0
native-country    0
income            0
dtype: int64

'?' counts per column:
age                  0
workclass         1836
fnlwgt               0
education            0
education-num        0
marital-status       0
occupation        1843
relationship         0
race                 0
sex                  0
capital-gain         0
capital-loss         0
hours-per-week       0
native-country     583
income               0
dtype: int64

Total missing (NaN + '?') per column:
workclass         1836
occupation        1843
native-country     583
dtype: int64


In [17]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

# Clean up missing values marked as "?"
adult_clean = adult.replace("?", pd.NA).dropna()

# Target: convert income to binary (0 = <=50K, 1 = >50K)
adult_clean["income"] = adult_clean["income"].str.strip()
y = (adult_clean["income"] == ">50K").astype(int)

# Features: one-hot encode categoricals, drop original income column
X = pd.get_dummies(adult_clean.drop(columns=["income"]), drop_first=True)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Fit the model
model = LogisticRegression(max_iter=1000)
model.fit(X_train_scaled, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [25]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

y_train_pred = model.predict(X_train_scaled)

print("Train Accuracy:", accuracy_score(y_train, y_train_pred))
print("Train Precision:", precision_score(y_train, y_train_pred))
print("Train Recall:", recall_score(y_train, y_train_pred))
print("Train F1-score:", f1_score(y_train, y_train_pred))

print("\nTrain Confusion Matrix:")
print(confusion_matrix(y_train, y_train_pred))

Train Accuracy: 0.8491441833478387
Train Precision: 0.7393767705382436
Train Recall: 0.6083916083916084
Train F1-score: 0.6675191815856778

Train Confusion Matrix:
[[16835  1288]
 [ 2352  3654]]


In [27]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

y_test_pred = model.predict(X_test_scaled)

print("Test Accuracy:", accuracy_score(y_test, y_test_pred))
print("Test Precision:", precision_score(y_test, y_test_pred))
print("Test Recall:", recall_score(y_test, y_test_pred))
print("Test F1-score:", f1_score(y_test, y_test_pred))

print("\nTest Confusion Matrix:")
print(confusion_matrix(y_test, y_test_pred))

Test Accuracy: 0.8473396320238687
Test Precision: 0.734085414987913
Test Recall: 0.6065246338215713
Test F1-score: 0.6642362376959533

Test Confusion Matrix:
[[4201  330]
 [ 591  911]]


In [43]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

rf_model = RandomForestClassifier(n_estimators=200, max_depth=20, random_state=10, n_jobs=-1)
rf_model.fit(X_train, y_train)

y_train_pred_rf = rf_model.predict(X_train)
y_test_pred_rf = rf_model.predict(X_test)

print("--- Random Forest: Train ---")
print("Accuracy:", accuracy_score(y_train, y_train_pred_rf))
print("Precision:", precision_score(y_train, y_train_pred_rf))
print("Recall:", recall_score(y_train, y_train_pred_rf))
print("F1-score:", f1_score(y_train, y_train_pred_rf))

print("\n--- Random Forest: Test ---")
print("Accuracy:", accuracy_score(y_test, y_test_pred_rf))
print("Precision:", precision_score(y_test, y_test_pred_rf))
print("Recall:", recall_score(y_test, y_test_pred_rf))
print("F1-score:", f1_score(y_test, y_test_pred_rf))

print("\nTest Confusion Matrix:")
print(confusion_matrix(y_test, y_test_pred_rf))

--- Random Forest: Train ---
Accuracy: 0.914169671349828
Precision: 0.9016125739946927
Recall: 0.7354312354312355
F1-score: 0.810087116001834

--- Random Forest: Test ---
Accuracy: 0.858942483010111
Precision: 0.7847769028871391
Recall: 0.5972037283621837
F1-score: 0.6782608695652174

Test Confusion Matrix:
[[4285  246]
 [ 605  897]]


In [ ]:
#Find optimal Max-Depth
#Found to be 20

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score
import pandas as pd

depth_values = [15, 18,20,22,24, 26,28,30]
results = []

for depth in depth_values:
    rf = RandomForestClassifier(
        n_estimators=200,
        max_depth=depth,
        random_state=42,
        n_jobs=-1
    )
    rf.fit(X_train, y_train)

    train_acc = accuracy_score(y_train, rf.predict(X_train))
    test_acc = accuracy_score(y_test, rf.predict(X_test))
    test_f1 = f1_score(y_test, rf.predict(X_test))

    results.append({
        "max_depth": depth,
        "train_accuracy": train_acc,
        "test_accuracy": test_acc,
        "test_f1": test_f1,
        "overfit_gap": train_acc - test_acc
    })

results_df = pd.DataFrame(results)
print(results_df)

best_row = results_df.loc[results_df["test_accuracy"].idxmax()]
print("\nBest max_depth based on test accuracy:")
print(best_row)

   max_depth  train_accuracy  test_accuracy   test_f1  overfit_gap
0         15        0.882175       0.854964  0.661770     0.027211
1         18        0.899913       0.857616  0.672262     0.042297
2         20        0.913797       0.858942  0.678261     0.054854
3         22        0.930913       0.856290  0.676130     0.074623
4         24        0.945087       0.856456  0.677107     0.088631
5         26        0.958142       0.855462  0.676077     0.102680
6         28        0.968461       0.855462  0.679412     0.113000
7         30        0.977952       0.853473  0.673560     0.124479

Best max_depth based on test accuracy:
max_depth         20.000000
train_accuracy     0.913797
test_accuracy      0.858942
test_f1            0.678261
overfit_gap        0.054854
Name: 2, dtype: float64
